# Data Drift Analysis

Compare the current production traffic of the credit-scoring API to the training reference set using [Evidently](https://docs.evidentlyai.com/). The notebook pulls real prediction logs from Supabase (a configured `DATABASE_URL` with accumulated traffic is required).

Outputs:
- `docs/drift_report.html` — full interactive Evidently report
- a narrative summary at the bottom of this notebook

In [ ]:
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.services.preprocessing import build_feature_dataframe  
from monitoring.drift_detection import (  
    generate_drift_report,
    load_reference_data,
    save_drift_report_html,
)

## 1. Reference data

Loaded from `monitoring/reference_data.csv` — a 1000-row sample of the training set used in Part 1, aligned with the 100 features the model expects.

In [ ]:
reference = load_reference_data()
print(f"reference shape: {reference.shape}")
reference.head(3)

## 2. Production data

The notebook pulls the most recent prediction logs from Supabase and projects each API input into the same 100-feature space using `build_feature_dataframe`. A configured `DATABASE_URL` with accumulated prediction traffic is required.

In [ ]:
def load_production_from_db(limit: int = 1000) -> pd.DataFrame:
    if not os.environ.get("DATABASE_URL"):
        raise RuntimeError("DATABASE_URL must be set to load production prediction logs.")
    from app.services.db_service import get_recent_predictions
    rows = get_recent_predictions(limit=limit)
    if not rows:
        raise RuntimeError("No prediction logs found in the database yet.")
    frames = [build_feature_dataframe(dict(r["input_features"])) for r in rows]
    return pd.concat(frames, ignore_index=True)


production = load_production_from_db()
print(f"Loaded {len(production)} rows from Supabase.")
print(f"production shape: {production.shape}")
production.head(3)

## 3. Evidently drift report

`generate_drift_report` runs the `DataDriftPreset` over every column that has at least one non-null value in both frames. The HTML is saved to `docs/drift_report.html`.

In [ ]:
snapshot = generate_drift_report(reference, production)
report_path = PROJECT_ROOT / "docs" / "drift_report.html"
report_path.parent.mkdir(parents=True, exist_ok=True)
save_drift_report_html(snapshot, str(report_path))
print(f"Drift report saved to {report_path}")

In [ ]:
html = snapshot.get_html_str(as_iframe=False)
display(HTML(html))